In [1]:
import pandas as pd
import numpy as np
import os
import shutil 
from PIL import Image
import matplotlib.pyplot as plt


In [2]:
india_df = pd.read_excel('data/raw/dataset anemia/India/India.xlsx')

print(india_df.shape)
print(india_df.head())
print(india_df['Hgb'].describe())

(95, 5)
   Number   Hgb Gender  Age  Note
0       1  12.2      M   29   NaN
1       2   8.0      F   36   NaN
2       3  10.7      F   30   NaN
3       4   8.3      F   39   NaN
4       5   7.8      F   29   NaN
count    95.000000
mean     11.471579
std       2.075678
min       7.600000
25%       9.950000
50%      11.300000
75%      12.800000
max      17.100000
Name: Hgb, dtype: float64


In [3]:
italy_df = pd.read_excel('data/raw/Dataset Anemia/Italy/Italy.xlsx')

print(italy_df.shape)
print(italy_df.head(10))
print(italy_df['Hgb'].describe())


(123, 9)
   Number   Hgb Gender  Age                               Note  Unnamed: 5  \
0       1   9.3      F   82  Forniceal conjunctiva not visible         NaN   
1       2  10.2      F   77       da segmentare la forniceale          NaN   
2       3  10.7      F   52       da segmentare la forniceale          NaN   
3       4  11.7      F   73       da segmentare la forniceale          NaN   
4       5  11.6      F   74       da segmentare la forniceale          NaN   
5       6    15      F   77       da segmentare la forniceale          NaN   
6       7    12      F   72       da segmentare la forniceale          NaN   
7       8  14.3      M   84       da segmentare la forniceale          NaN   
8       9   9.8      M   61       da segmentare la forniceale          NaN   
9      10  12.2      M   70       da segmentare la forniceale          NaN   

  Unnamed: 6 Unnamed: 7             Unnamed: 8  
0        NaN        NaN                    NaN  
1        NaN        NaN  Segmentat

In [4]:
# Force numeric conversion first
india_df['Hgb'] = pd.to_numeric(india_df['Hgb'], errors='coerce')
italy_df['Hgb'] = pd.to_numeric(italy_df['Hgb'], errors='coerce')

# Drop missing Hgb values
india_df = india_df.dropna(subset=['Hgb'])
italy_df = italy_df.dropna(subset=['Hgb'])

# Apply severity labels
def hb_val_to_severity(hb):
    if hb < 8:
        return 'Severe'
    elif hb < 11:
        return 'Moderate'
    elif hb < 12:
        return 'Mild'
    else:
        return 'None'

india_df['Severity'] = india_df['Hgb'].apply(hb_val_to_severity)
italy_df['Severity'] = italy_df['Hgb'].apply(hb_val_to_severity)

print("India severity counts:")
print(india_df['Severity'].value_counts())
print("\nItaly severity counts:")
print(italy_df['Severity'].value_counts())

India severity counts:
Severity
None        39
Moderate    38
Mild        16
Severe       2
Name: count, dtype: int64

Italy severity counts:
Severity
None        87
Moderate    13
Mild         6
Severe       1
Name: count, dtype: int64


In [5]:
print(india_df.shape)
print(italy_df.shape)
print(india_df['Severity'].value_counts)
print(italy_df['Severity'].value_counts)

(95, 6)
(107, 10)
<bound method IndexOpsMixin.value_counts of 0         None
1     Moderate
2     Moderate
3     Moderate
4       Severe
        ...   
90        None
91        None
92        None
93        Mild
94        None
Name: Severity, Length: 95, dtype: str>
<bound method IndexOpsMixin.value_counts of 0      Moderate
1      Moderate
2      Moderate
3          Mild
4          Mild
         ...   
118        None
119        None
120        None
121        None
122        None
Name: Severity, Length: 107, dtype: str>


In [ ]:
import matplotlib.pyplot as plt

india_counts = india_df['Severity'].value_counts()
italy_counts = italy_df['Severity'].value_counts()

print("India counts:", india_counts)
print("Italy counts:", italy_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(india_counts.index, india_counts.values, 
            color=['red', 'orange', 'yellow', 'green'])
axes[0].set_title('Severity Distribution in India')
axes[0].set_xlabel('Severity')
axes[0].set_ylabel('Count')

axes[1].bar(italy_counts.index, italy_counts.values,
            color=['red', 'orange', 'yellow', 'green'])
axes[1].set_title('Severity Distribution in Italy')
axes[1].set_xlabel('Severity')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('assets/severity_distribution.png')
plt.show()

In [5]:
import os 
patient_folder = 'data/raw/dataset anemia/India/1'

files = os.listdir(patient_folder)

print(files)

['20200118_164733.jpg', '20200118_164733_forniceal.png', '20200118_164733_forniceal_palpebral.png', '20200118_164733_palpebral.png']


In [10]:
from IPython.display import display
from PIL import Image
import os

img_path = 'data/raw/dataset anemia/India/1/'
files = os.listdir(img_path)
print("Files in patient 1 folder:", files)

# Show the palpebral image with error handling
for f in files:
    if 'palpebral' in f and 'forniceal' not in f:
        full_path = os.path.join(img_path, f)
        try:
            img = Image.open(full_path)
            img.verify()  # verify it's a real image
            img = Image.open(full_path)  # reopen after verify
            display(img)
            print(f"Image: {f}, Size: {img.size}")
        except Exception as e:
            print(f"Skipping corrupted file: {f} — {e}")

Files in patient 1 folder: ['20200118_164733.jpg', '20200118_164733_forniceal.png', '20200118_164733_forniceal_palpebral.png', '20200118_164733_palpebral.png']
Skipping corrupted file: 20200118_164733_palpebral.png — cannot identify image file 'data/raw/dataset anemia/India/1/20200118_164733_palpebral.png'


In [14]:
# Identifying and Removing all corrupted images from the dataset 

from PIL import Image
import os

corrupted = []
india_path = 'data/raw/dataset anemia/India'

for folder in os.listdir(india_path):
    folder_path = os.path.join(india_path, folder)

    if os.path.isdir(folder_path):
        for f in os.listdir(folder_path):
            if f.endswith('.png') or f.endswith('.jpg'):
                try: 
                    img = Image.open(os.path.join(folder_path, f))
                    img.verify()
                except Exception as e: 
                    corrupted.append(f)

print('Total number of corrupted files : ', len(corrupted))
print('Corrupted Files : ', corrupted)

corrupted_png = [f for f in corrupted if f.endswith('.png')]
corrupted_jpg = [f for f in corrupted if f.endswith('.jpg')]

print(len(corrupted_jpg))
print(len(corrupted_png))


Total number of corrupted files :  96
Corrupted Files :  ['20200118_164733_forniceal.png', '20200118_164733_forniceal_palpebral.png', '20200118_164733_palpebral.png', '20200203_091841_forniceal.png', '20200203_091841_forniceal_palpebral.png', '20200203_091841_palpebral.png', '20200203_190841_forniceal.png', '20200203_190841_forniceal_palpebral.png', '20200203_190841_palpebral.png', '20200204_155221_forniceal.png', '20200204_155221_forniceal_palpebral.png', '20200204_155221_palpebral.png', '20200211_140525_forniceal.png', '20200211_140525_forniceal_palpebral.png', '20200211_140525_palpebral.png', '20200124_154320_forniceal.png', '20200124_154320_forniceal_palpebral.png', '20200124_154320_palpebral.png', '20200211_150240_forniceal.png', '20200211_150240_forniceal_palpebral.png', '20200211_150240_palpebral.png', '20200213_121216_forniceal.png', '20200213_121216_forniceal_palpebral.png', '20200213_121216_palpebral.png', '20200213_150536_forniceal.png', '20200213_150536_forniceal_palpebral.